# Cross-Entropy

Cross-entropy is a way to measure **how wrong** our model's predictions are.

Think of it like a scoring system:
- If our model is **confident and correct** → low penalty (small error)
- If our model is **confident and wrong** → huge penalty (large error)

It answers the question: *"How surprised are we by the prediction, given the true answer?"*

### Why use logarithms?

The `ln` (natural log) is what creates the "punishment" effect:
- `ln(1.0) = 0` → no penalty when prediction matches reality perfectly
- `ln(0.5) ≈ -0.69` → moderate penalty for an uncertain prediction
- `ln(0.01) ≈ -4.6` → severe penalty for a very wrong prediction

The negative sign at the front flips everything so the error is always positive.

$$
\text{Cross-Entropy} = -\sum_{i=1}^{m} \left[ y_i \ln(p_i) + (1 - y_i)\ln(1 - p_i) \right]
$$


### Breaking down the formula

For **binary classification** (only 2 possible outcomes: yes/no, 0/1):

| Part | What it does |
|------|-------------|
| $y_i$ | The true label (0 or 1) for sample $i$ |
| $p_i$ | Our model's predicted probability that sample $i$ is class 1 |
| $y_i \ln(p_i)$ | Penalty when the true label is 1 — checks how close $p_i$ is to 1 |
| $(1 - y_i)\ln(1 - p_i)$ | Penalty when the true label is 0 — checks how close $p_i$ is to 0 |
| $-\sum$ | Sum all penalties and flip sign to make the total positive |

**Key insight:** For each sample, only ONE of the two terms is active (because $y_i$ is either 0 or 1), so the formula naturally picks the right penalty.

### Implementation (step-by-step version)

The function below computes cross-entropy by breaking it into small, readable steps:

In [1]:
import numpy as np

def cross_entropy(Y, P):
    """
    This function takes as input two lists Y and P, and returns the float
    corresponding to their cross-entropy.
    """
    
    P_log = np.log(P)
    P_log_times_Y = np.multiply(P_log, Y)
    
    P_log_wrong = np.log(1 - P)
    Y_wrong = 1 - Y
    P_log_times_Y_wrong = np.multiply(P_log_wrong, Y_wrong)
    
    return -1 * (P_log_times_Y + P_log_times_Y_wrong)

**What each line does:**

1. `np.log(P)` → Compute $\ln(p_i)$ for each prediction
2. `np.multiply(P_log, Y)` → Multiply by true labels → gives us the $y_i \ln(p_i)$ term
3. `np.log(1 - P)` → Compute $\ln(1 - p_i)$ for each prediction
4. `1 - Y` → Flip labels (0 becomes 1, 1 becomes 0)
5. `np.multiply(P_log_wrong, Y_wrong)` → gives us the $(1-y_i)\ln(1-p_i)$ term
6. Multiply by -1 and add both terms together

**Note:** This version returns the error *per sample* (no sum), which is useful for seeing individual contributions.

## Solution

The solution below does the exact same thing in one line using NumPy broadcasting:

- `np.float_(Y)` and `np.float_(P)` → ensure inputs are floats (avoids integer math issues)
- `Y * np.log(P)` → the $y_i \ln(p_i)$ term
- `(1 - Y) * np.log(1 - P)` → the $(1-y_i)\ln(1-p_i)$ term
- `np.sum(...)` → adds up all individual errors into one total number
- `-` at the front → flips the sign to make it positive

This version returns the **total** cross-entropy (a single scalar), not per-sample values.

In [ ]:
def cross_entropy(Y, P):
    Y = np.float_(Y)
    P = np.float_(P)
    return -np.sum(Y  *np.log(P) + (1 - Y)*  np.log(1 - P))

## More than one class

$$
\text{Cross-Entropy} = -\sum_{i=1}^{n}\sum_{j=1}^{m} y_{ij}\ln(p_{ij})
$$


### Multiclass explained

When there are **more than 2 classes** (e.g., cat/dog/bird), we can't just use 0 and 1. Instead:

- $y_{ij}$ = 1 if sample $i$ belongs to class $j$, else 0 (this is called **one-hot encoding**)
- $p_{ij}$ = our model's predicted probability that sample $i$ is class $j$

The formula sums over all samples ($i$) AND all classes ($j$). For each sample, only the term where $y_{ij} = 1$ contributes (the rest are zeroed out), so we're essentially asking: *"How confident was the model about the correct class?"*

**Example:** If sample 1 is a "dog" (class 2 out of 3), and our model predicts `[0.1, 0.7, 0.2]`:
- Only the $j=2$ term matters: $1 \times \ln(0.7) \approx -0.36$
- The model was fairly confident → small penalty